# Adversarial Training for Deep Hedging — Heston Model

Replicates He, Sutter & Gonon (2025) **Sections 5.2–5.3**: adversarial training
improves out-of-sample and out-of-distribution hedging performance.

**Three methods compared:**
- `Clean` — standard deep hedging (minimise CVaR loss on clean paths)
- `S-Attack` — adversarial training: perturb stock price paths only
- `SV-Attack` — adversarial training: jointly perturb stock + variance paths,
  rebuild VarPrice from V_att via Heston closed-form

**Training procedure (two-stage):**
1. Pre-train on clean paths for `N_EPOCHS_CLEAN` epochs
2. Fine-tune with combined loss $\mathcal{L}_{\text{adv}}(\theta) = \alpha\,\mathcal{L}_{\text{clean}} + \mathcal{L}_{\text{adv paths}}$ for `N_EPOCHS_ADV` epochs

Clean baseline uses the same total epoch budget (= `N_EPOCHS_CLEAN + N_EPOCHS_ADV`) for a fair comparison.

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set QUICK_RUN=False for the full experiment matching the paper (~15–30 min).
# QUICK_RUN=True runs a reduced experiment suitable for verification (~3–5 min).
QUICK_RUN = True

if QUICK_RUN:
    N_SIZES        = [5_000, 20_000, 100_000]
    N_SEEDS        = 2       # partitions per N
    N_EPOCHS_CLEAN = 100     # Stage 1: clean pre-training
    N_EPOCHS_ADV   = 60      # Stage 2: adversarial fine-tuning
    N_OOD_CFGS    = 20       # perturbed Heston configs for OOD eval
else:
    N_SIZES        = [5_000, 10_000, 20_000, 50_000, 100_000]
    N_SEEDS        = 3
    N_EPOCHS_CLEAN = 300
    N_EPOCHS_ADV   = 400
    N_OOD_CFGS    = 50

M_TRAIN    = 100_000   # total training paths; partitioned into blocks of size N
M_TEST     = 1_000_000   # OOS test set (paper uses 1 M; 100 k is tractable)
M_OOD      = 10_000    # paths per OOD config

BATCH_SIZE = 10_000
LR         = 5e-3      # single learning rate; 4-level schedule decays proportionally
ALPHA_BAL  = 1.0       # alpha in L_adv = alpha*L_clean + L_adv_paths  (paper eq. 5.1)

# Attack hyper-parameters
ATK_DELTA   = 0.1      # Wasserstein ball radius (default; swept in validation cell)
ATK_RATIO   = 4.0      # step-size factor; alpha = delta*ratio/n
ATK_N_TRAIN = 5       # budget attack iterations during training (matches reference hardcoded 20)
ATK_N_EVAL  = 20       # iterations at evaluation (accurate)

# Loss / option parameters
ALPHA_CVAR = 0.5
K          = 100.0
N_STEPS    = 30

# Heston model parameters — identical to src/train_heston.py
S0    = 100.0
v0    = 0.04
kappa = 1.0
theta = 0.04
xi    = 2.0
rho   = -0.7
T     = 1.0

# Validation sweep grids (used in the sweep cell)
DELTA_SWEEP      = [0.05, 0.10, 0.20]
ALPHA_CVAR_SWEEP = [0.90, 0.95, 0.99]

In [2]:
# ── Imports & Device ───────────────────────────────────────────────────────────
import sys
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, '..')
from src.heston_simulator import (
    HestonParams, HestonSimulator, OutOfSampleHestonSimulator
)
from src.hedging.hedge_network import HestonHedgeNet
from src.hedging.loss import HestonCVaRLoss

torch.set_float32_matmul_precision('high')

device = (
    torch.device('mps')  if torch.backends.mps.is_available()  else
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('cpu')
)
print(f"Device : {device}")
print(f"QUICK_RUN={QUICK_RUN}  N_SIZES={N_SIZES}  N_SEEDS={N_SEEDS}")
print(f"Epochs  : {N_EPOCHS_CLEAN} clean + {N_EPOCHS_ADV} adv = "
      f"{N_EPOCHS_CLEAN + N_EPOCHS_ADV} total per run")

Device : mps
QUICK_RUN=True  N_SIZES=[5000, 20000, 100000]  N_SEEDS=2
Epochs  : 100 clean + 60 adv = 160 total per run


In [ ]:
# ── Simulate Paths ─────────────────────────────────────────────────────────────
print("Simulating training paths (100 k, seed=19) ...")
t0 = time.perf_counter()
base_params = HestonParams(
    S0=S0, v0=v0, kappa=kappa, theta=theta, xi=xi, rho=rho,
    T=T, N=N_STEPS, M=M_TRAIN,
)
S_tr, V_tr, VP_tr = HestonSimulator(base_params).simulate(seed=19)
print(f"  Done in {time.perf_counter()-t0:.1f}s   shape: {S_tr.shape}")
print(f"  VP_tr[:, 0] mean = {VP_tr[:, 0].mean():.6f}  (raw units, before scaling)")

print("\nSimulating test paths (1m, seed=42) ...")
t0 = time.perf_counter()
test_params = HestonParams(
    S0=S0, v0=v0, kappa=kappa, theta=theta, xi=xi, rho=rho,
    T=T, N=N_STEPS, M=M_TEST,
)
S_te, V_te, VP_te = HestonSimulator(test_params).simulate(seed=42)
print(f"  Done in {time.perf_counter()-t0:.1f}s")

# Scale VP so that dVP ≈ dS in magnitude.
# Without scaling: dVP ≈ 0.008, dS ≈ 0.8 → 100× disparity → network never learns variance hedge.
VP_SCALE = 100.0
VP_tr = VP_tr * VP_SCALE
VP_te = VP_te * VP_SCALE
print(f"\nVP scaled by {VP_SCALE:.0f}:  dVP (scaled) ≈ {float((VP_te[:,1:]-VP_te[:,:-1]).abs().mean())*1:.4f}")

# p0 warm-start: VaR_α(C_T) = α-quantile — the optimal starting point for CVaR dual variable.
# Using mean(C_T) instead would be far above optimal p0* ≈ 0, killing gradient signal early.
with torch.no_grad():
    C_T_tr = torch.clamp(S_tr[:, -1] - K, min=0.0)
    p0_init_global = float(C_T_tr.quantile(ALPHA_CVAR))
print(f"VaR_{ALPHA_CVAR:.2f}(C_T) = {p0_init_global:.4f}  (mean C_T = {float(C_T_tr.mean()):.4f})")

# Move training data to device once; slice on-device during training loops
S_tr  = S_tr.to(device)
V_tr  = V_tr.to(device)
VP_tr = VP_tr.to(device)
print(f"Training tensors moved to {device}.")

In [4]:
print(f"S_te[:,1:] - S_te[:,:-1]  mean abs = {(S_te[:,1:] - S_te[:,:-1]).abs().mean():.6f}")
print(f"VP_te[:,1:] - VP_te[:,:-1] mean abs = {(VP_te[:,1:] - VP_te[:,:-1]).abs().mean():.6f}")
print(f"S_te[:, -1] mean = {S_te[:,-1].mean():.4f}")
print(f"C_T mean = {torch.clamp(S_te[:,-1] - K, min=0).mean():.4f}")
print(f"T={T},  dt={T/N_STEPS:.6f}")

S_te[:,1:] - S_te[:,:-1]  mean abs = 0.798612
VP_te[:,1:] - VP_te[:,:-1] mean abs = 0.007927
S_te[:, -1] mean = 100.0037
C_T mean = 3.7190
T=1.0,  dt=0.033333


In [ ]:
# ── Helper Functions ───────────────────────────────────────────────────────────
loss_fn = HestonCVaRLoss(K=K, alpha=ALPHA_CVAR)


def make_input(S: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """(batch, N+1) x 2 -> (batch, N, 2) = [log S_t, V_t] for t=0..N-1."""
    return torch.cat([
        torch.log(S[:, :-1]).unsqueeze(-1),
        V[:, :-1].unsqueeze(-1),
    ], dim=-1)


def V_to_VarPrice(V: torch.Tensor) -> torch.Tensor:
    """
    Vectorized Heston closed-form variance swap price, scaled by VP_SCALE.
    Differentiable in V. Must match the scaling applied to VP_tr/VP_te in Cell 3.
    """
    dt = T / N_STEPS
    VP = torch.zeros(V.shape[0], V.shape[1], device=V.device, dtype=V.dtype)
    VP[:, 0] = (V[:, 0] - theta) / kappa * (1.0 - np.exp(-kappa * T)) + theta * T
    var_int = 0.5 * dt * (V[:, :-1] + V[:, 1:]).cumsum(dim=1)
    time_factors = torch.arange(1, N_STEPS + 1, device=V.device, dtype=V.dtype) * dt
    correction = (
        (V[:, 1:] - theta) / kappa * (1.0 - torch.exp(-kappa * (T - time_factors)))
        + theta * (T - time_factors)
    )
    VP[:, 1:] = var_int + correction
    return VP * VP_SCALE


def _attack_loss_fn(holding: torch.Tensor, S: torch.Tensor, VP: torch.Tensor) -> torch.Tensor:
    """
    CVaR loss with p0 computed analytically as quantile(X, alpha).
    Matches loss_CVAR(p0_mode='search') from the reference implementation.
    Used inside attack loops only; training uses HestonCVaRLoss with learned p0.
    """
    dS  = S[:, 1:] - S[:, :-1]
    dVP = VP[:, 1:] - VP[:, :-1]
    PnL = (holding[:, :, 0] * dS + holding[:, :, 1] * dVP).sum(1)
    X   = torch.clamp(S[:, -1] - K, min=0.0) - PnL
    p0  = X.quantile(ALPHA_CVAR).item()
    return torch.clamp(X - p0, min=0.0).mean() / (1.0 - ALPHA_CVAR) + p0


@torch.no_grad()
def compute_hedging_errors(
    net: nn.Module,
    S: torch.Tensor,
    V: torch.Tensor,
    VarPrice: torch.Tensor,
    batch_size: int = 10_000,
) -> torch.Tensor:
    """Returns (M,) CPU tensor of per-path hedging errors X = C_T - PnL."""
    net = net.to(device).eval()
    errors = []
    for i in range(0, S.shape[0], batch_size):
        S_b   = S[i:i+batch_size].to(device)
        V_b   = V[i:i+batch_size].to(device)
        VP_b  = VarPrice[i:i+batch_size].to(device)
        h     = net(make_input(S_b, V_b))
        dS    = S_b[:, 1:] - S_b[:, :-1]
        dVP   = VP_b[:, 1:] - VP_b[:, :-1]
        PnL   = (h[:, :, 0] * dS + h[:, :, 1] * dVP).sum(1)
        X     = loss_fn.terminal_payoff(S_b[:, -1]) - PnL
        errors.append(X.cpu())
    return torch.cat(errors)


def empirical_ES(errors: torch.Tensor, alpha: float = ALPHA_CVAR) -> float:
    """Empirical ES_alpha: mean of the top (1-alpha) fraction of losses."""
    k = max(1, int(np.ceil((1.0 - alpha) * len(errors))))
    return float(torch.topk(errors, k).values.mean())


def make_lr_lambda(n_epochs: int):
    """LR schedule proportional to training length — works for any epoch count."""
    def _fn(epoch: int) -> float:
        if epoch < n_epochs // 2:       return 1.0
        if epoch < 3 * n_epochs // 4:   return 0.1
        return 0.01
    return _fn


print("Helper functions defined.")

In [6]:
# ── S-Only Budget Attack ────────────────────────────────────────────────────────
# Inlined from Heston_Attacker.S_budget_attack (reference Heston_util.py).
# Perturbs S only; V and VarPrice unchanged.

def s_budget_attack(network, S, V, delta, ratio, iters):
    """Returns (S_att, V_unchanged, VP_unchanged)."""
    if delta == 0:
        return S, V, V_to_VarPrice(V)

    def _perf(S_a, V_a):
        VP_a = V_to_VarPrice(V_a)
        with torch.no_grad():
            h = network(make_input(S_a, V_a)).squeeze()
        dS  = S_a[:, 1:] - S_a[:, :-1]
        dVP = VP_a[:, 1:] - VP_a[:, :-1]
        PnL = (h[:, :, 0] * dS + h[:, :, 1] * dVP).sum(1)
        X   = torch.clamp(S_a[:, -1] - K, min=0.0) - PnL
        p0  = X.quantile(ALPHA_CVAR).item()
        return (torch.clamp(X - p0, min=0.0).mean() / (1.0 - ALPHA_CVAR) + p0).item()

    budget = torch.ones(S.shape[0], 2, device=S.device) * delta
    budget.requires_grad = True
    att_sign = torch.ones(S.shape[0], S.shape[1], 2, device=S.device)
    att_sign[:, 0, :] = 0
    att_sign.requires_grad = True
    alpha_step  = delta * ratio / iters
    att_best    = (budget.unsqueeze(1) * att_sign).clone().detach()
    result_best = 0.0

    for _i in range(iters):
        S_att = S + budget[:, 0].unsqueeze(1) * att_sign[:, :, 0]
        V_att = V
        perf  = _perf(S_att.detach(), V_att.detach())
        if perf > result_best:
            att_best    = (budget.unsqueeze(1) * att_sign).clone().detach()
            result_best = _perf(S_att.detach(), V_att.detach())
        VP_att  = V_to_VarPrice(V_att)
        holding = network(make_input(S_att, V_att)).squeeze()
        loss    = _attack_loss_fn(holding, S_att, VP_att)
        grad_b  = torch.autograd.grad(loss, budget,   retain_graph=True)[0]
        grad_a  = torch.autograd.grad(loss, att_sign, retain_graph=True)[0]
        with torch.no_grad():
            budget_new = budget + alpha_step * grad_b / (grad_b.pow(2).mean() + 1e-10).sqrt()
            budget_new = budget_new / budget_new.square().mean().sqrt() * delta / 1.414
            budget_new = torch.max(budget_new, torch.zeros_like(budget_new))
            budget.copy_(budget_new)
            grad_a[:, 0, :] = 0
            att_sign.copy_(grad_a.sign())

    S_att = S + budget[:, 0].unsqueeze(1) * att_sign[:, :, 0]
    if _perf(S_att.detach(), V.detach()) > result_best:
        att_best = (budget.unsqueeze(1) * att_sign).clone().detach()
    S_att = (S + att_best[:, :, 0]).detach()
    return S_att, V.clone(), V_to_VarPrice(V.clone())


print("S-only budget attack defined.")

S-only budget attack defined.


In [7]:
# ── Joint SV Budget Attack ──────────────────────────────────────────────────────
# Inlined from Heston_Attacker.SV_budget_attack (reference Heston_util.py).
# Perturbs S and V jointly; VarPrice rebuilt from V_att via V_to_VarPrice.
# Sign update: soft momentum clamp — att_sign_old is a reference (not clone),
# so momentum term 0.25*(att_sign - att_sign_old) is always 0 (matches reference).

def sv_budget_attack(network, S, V, delta, ratio, iters):
    """Returns (S_att, V_att, VP_att)."""
    if delta == 0:
        return S, V, V_to_VarPrice(V)

    def _perf(S_a, V_a):
        VP_a = V_to_VarPrice(V_a)
        with torch.no_grad():
            h = network(make_input(S_a, V_a)).squeeze()
        dS  = S_a[:, 1:] - S_a[:, :-1]
        dVP = VP_a[:, 1:] - VP_a[:, :-1]
        PnL = (h[:, :, 0] * dS + h[:, :, 1] * dVP).sum(1)
        X   = torch.clamp(S_a[:, -1] - K, min=0.0) - PnL
        p0  = X.quantile(ALPHA_CVAR).item()
        return (torch.clamp(X - p0, min=0.0).mean() / (1.0 - ALPHA_CVAR) + p0).item()

    budget = torch.ones(S.shape[0], 2, device=S.device) * delta
    budget.requires_grad = True
    att_sign = torch.ones(S.shape[0], S.shape[1], 2, device=S.device)
    att_sign[:, 0, :] = 0
    att_sign_old = att_sign.clone().detach()
    att_sign.requires_grad = True
    alpha_step  = delta * ratio / iters
    att_best    = (budget.unsqueeze(1) * att_sign).clone().detach()
    result_best = 0.0

    for _i in range(iters):
        S_att = S + budget[:, 0].unsqueeze(1) * att_sign[:, :, 0]
        V_att = V + budget[:, 1].unsqueeze(1) * att_sign[:, :, 1] / 100
        perf  = _perf(S_att.detach(), V_att.detach())
        if perf > result_best:
            att_best    = (budget.unsqueeze(1) * att_sign).clone().detach()
            result_best = _perf(S_att.detach(), V_att.detach())
        VP_att  = V_to_VarPrice(V_att)
        holding = network(make_input(S_att, V_att)).squeeze()
        loss    = _attack_loss_fn(holding, S_att, VP_att)
        grad_b  = torch.autograd.grad(loss, budget,   retain_graph=True)[0]
        grad_a  = torch.autograd.grad(loss, att_sign, retain_graph=True)[0]
        with torch.no_grad():
            budget_new = budget + alpha_step * grad_b / (grad_b.pow(2).mean() + 1e-10).sqrt()
            budget_new = budget_new / budget_new.square().mean().sqrt() * delta / 1.414
            budget_new = torch.max(budget_new, torch.zeros_like(budget_new))
            budget.copy_(budget_new)
            att_sign_new = (att_sign + 0.75 * grad_a.sign() + 0.25 * (att_sign - att_sign_old)).clamp(-1, 1)
            att_sign_new[:, 0, :] = 0
            att_sign_old = att_sign          # reference assignment — momentum = 0 (matches reference)
            att_sign.copy_(att_sign_new)

    S_att = S + budget[:, 0].unsqueeze(1) * att_sign[:, :, 0]
    V_att = V + budget[:, 1].unsqueeze(1) * att_sign[:, :, 1] / 100
    if _perf(S_att.detach(), V_att.detach()) > result_best:
        att_best = (budget.unsqueeze(1) * att_sign).clone().detach()
    S_att  = (S + att_best[:, :, 0]).detach()
    V_att  = (V + att_best[:, :, 1] / 100).detach()
    VP_att = V_to_VarPrice(V_att)
    return S_att, V_att, VP_att


print("SV-joint budget attack defined.")

SV-joint budget attack defined.


In [8]:
# ── Shared Attack Wrapper ─────────────────────────────────────────────────────

def run_attack(network, S, V, VP, attack_method, delta, ratio, n):
    """
    Returns (S_att, V_att, VP_att).
    attack_method: 'S' — stock only  |  'SV' — joint stock + variance
    VP is accepted for API compatibility but not forwarded — budget attacks
    reconstruct VarPrice from V internally.
    """
    if attack_method == 'S':
        return s_budget_attack(network, S, V, delta, ratio, n)
    elif attack_method == 'SV':
        return sv_budget_attack(network, S, V, delta, ratio, n)
    else:
        raise ValueError(f"Unknown attack_method '{attack_method}': choose 'S' or 'SV'")


print("Attack wrapper defined.")

Attack wrapper defined.


In [ ]:
# ── Training Function ───────────────────────────────────────────────────────────
from torch.utils.data import TensorDataset, DataLoader


def train_heston(
    S: torch.Tensor,
    V: torch.Tensor,
    VP: torch.Tensor,
    attack_method: str | None,   # None → clean only, 'S' or 'SV' → adversarial
    alpha_cvar: float = ALPHA_CVAR,
    delta: float = ATK_DELTA,
    desc: str = "",
) -> tuple[nn.Module, float]:
    """
    Unified training loop matching reference training script.
      Epochs 0 .. N_EPOCHS_CLEAN-1   : clean CVaR loss with p0_clean
      Epochs N_EPOCHS_CLEAN .. total-1: ALPHA_BAL*L_clean + L_att with p0_att
                                        (attack_method=None → clean throughout)
    LR schedule: 4-level decay at proportions [28.6%, 71.4%, 85.7%] of total epochs,
    matching reference thresholds [200, 500, 600] / 700.
    Returns (network_cpu, p0_clean_scalar).
    """
    tfn      = HestonCVaRLoss(K=K, alpha=alpha_cvar)
    net      = HestonHedgeNet(N=N_STEPS, width=20).to(device)
    p0_clean = nn.Parameter(torch.tensor(p0_init_global, device=device))
    p0_att   = nn.Parameter(torch.tensor(p0_init_global, device=device))

    n_total = N_EPOCHS_CLEAN + (N_EPOCHS_ADV if attack_method else 0)
    t1 = int(0.286 * n_total)
    t2 = int(0.714 * n_total)
    t3 = int(0.857 * n_total)

    opt = torch.optim.Adam(
        [{'params': net.parameters()}, {'params': [p0_clean, p0_att]}], lr=LR
    )
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lr_lambda=lambda e: 1.0 if e < t1 else 0.1 if e < t2 else 0.01 if e < t3 else 0.001,
    )
    loader = DataLoader(TensorDataset(S, V, VP), batch_size=BATCH_SIZE, shuffle=True)
    prefix = f"{desc} " if desc else ""

    pbar = tqdm(
        range(n_total), desc=f"{prefix}train", leave=False,
        bar_format=(
            "{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
            "  loss_c={postfix[0]:.4f}  loss_a={postfix[1]:.4f}  p0={postfix[2]:.4f}"
        ),
        postfix=[0.0, 0.0, p0_init_global],
    )

    for epoch in pbar:
        adv_phase = (attack_method is not None) and (epoch >= N_EPOCHS_CLEAN)

        if adv_phase and epoch == N_EPOCHS_CLEAN:
            p0_att.data.copy_(p0_clean.detach())

        last_loss_c = last_loss_a = 0.0
        net.train()
        for S_b, V_b, VP_b in loader:
            h_c    = net(make_input(S_b, V_b))
            loss_c = tfn(h_c, S_b, VP_b, p0_clean)

            if adv_phase:
                S_att, V_att, VP_att = run_attack(
                    net, S_b, V_b, VP_b, attack_method, delta, ATK_RATIO, ATK_N_TRAIN
                )
                h_a    = net(make_input(S_att, V_att))
                loss_a = tfn(h_a, S_att, VP_att, p0_att)
                total  = ALPHA_BAL * loss_c + loss_a
                last_loss_a = loss_a.item()
            else:
                total = loss_c

            opt.zero_grad()
            total.backward()
            opt.step()
            last_loss_c = loss_c.item()

        sched.step()
        pbar.postfix[0] = last_loss_c
        pbar.postfix[1] = last_loss_a
        pbar.postfix[2] = p0_clean.item()

    return net.cpu(), float(p0_clean.item())


print("Training function defined.")

In [17]:
# ── Main Experiment Loop ───────────────────────────────────────────────────────
results_oos = {}
stored_nets = {}

METHODS    = ['clean', 'S', 'SV']
total_runs = sum(min(M_TRAIN // N, N_SEEDS) for N in N_SIZES) * len(METHODS)
run_idx    = 0

print(f"Starting experiment: {total_runs} training runs\n")
t_exp = time.perf_counter()

outer_bar = tqdm(total=total_runs, desc="runs", unit="run",
                 bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]  {postfix}",
                 postfix="")

for N in N_SIZES:
    n_runs = min(M_TRAIN // N, N_SEEDS)
    for s in range(n_runs):
        S_n  = S_tr[s * N : (s + 1) * N]
        V_n  = V_tr[s * N : (s + 1) * N]
        VP_n = VP_tr[s * N : (s + 1) * N]

        for method in METHODS:
            run_idx  += 1
            run_label = f"N={N} {method} s{s}"
            outer_bar.set_postfix_str(run_label)
            t0 = time.perf_counter()

            net, p0_val = train_heston(
                S_n, V_n, VP_n,
                attack_method=None if method == 'clean' else method,
                alpha_cvar=ALPHA_CVAR,
                delta=ATK_DELTA,
                desc=run_label,
            )

            # ── Gradient sanity check (add inside train_heston instead — see below) ──

            # ── In-sample sanity check ──────────────────────────────────────────────
            errors_train = compute_hedging_errors(net, S_n.cpu(), V_n.cpu(), VP_n.cpu())
            print(f"  [SANITY] p0_clean = {p0_val:.4f}")
            print(f"  [SANITY] Train Mean Error = {errors_train.mean():.4f}  "
                f"(should be < C_T mean 1.76 if learning)")
            print(f"  [SANITY] Train ES_0.5     = {empirical_ES(errors_train, 0.5):.4f}")

            # Deep diagnostic
            net_d = net.to(device).eval()
            with torch.no_grad():
                S_b = S_te[:500].to(device)
                V_b = V_te[:500].to(device)
                VP_b = VP_te[:500].to(device)
                
                h = net_d(make_input(S_b, V_b))
                dS  = S_b[:, 1:] - S_b[:, :-1]
                dVP = VP_b[:, 1:] - VP_b[:, :-1]
                
                pnl_S  = (h[:, :, 0] * dS).sum(1)
                pnl_VP = (h[:, :, 1] * dVP).sum(1)
                C_T    = torch.clamp(S_b[:, -1] - K, min=0.0)
                
                print(f"  h[:,:,0] mean={h[:,:,0].mean():.4f} std={h[:,:,0].std():.4f}  (stock holdings)")
                print(f"  h[:,:,1] mean={h[:,:,1].mean():.4f} std={h[:,:,1].std():.4f}  (varswap holdings)")
                print(f"  PnL_S   mean={pnl_S.mean():.4f}  std={pnl_S.std():.4f}")
                print(f"  PnL_VP  mean={pnl_VP.mean():.4f}  std={pnl_VP.std():.4f}")
                print(f"  C_T     mean={C_T.mean():.4f}  std={C_T.std():.4f}")
                print(f"  dVP     mean abs={dVP.abs().mean():.6f}")
                print(f"  dS      mean abs={dS.abs().mean():.6f}")

            errors  = compute_hedging_errors(net, S_te, V_te, VP_te)
            es_val  = empirical_ES(errors, alpha=0.5)
            results_oos[(N, method, s)] = es_val
            stored_nets[(N, method, s)] = net

            outer_bar.update(1)
            tqdm.write(f"[{run_idx:2d}/{total_runs}] {run_label:22s}  "
                       f"Mean Error={errors.mean().item():.4f}  ({time.perf_counter()-t0:.0f}s)")

outer_bar.close()
print(f"\nAll training done in {(time.perf_counter()-t_exp)/60:.1f} min")

Starting experiment: 15 training runs



runs:   0%|          | 0/15 [00:00<?]  

N=5000 clean s0 train:   0%|          | 0/100 [00:00<?, ?it/s]  loss_c=0.0000  loss_a=0.0000  p0=3.7246

  [GRAD] last clean epoch grad norm = 6.758639  p0_clean = 3.5538
  [SANITY] p0_clean = 3.5538
  [SANITY] Train Mean Error = 2.0174  (should be < C_T mean 1.76 if learning)
  [SANITY] Train ES_0.5     = 4.4165
  h[:,:,0] mean=0.3238 std=0.3735  (stock holdings)
  h[:,:,1] mean=0.0863 std=0.7935  (varswap holdings)
  PnL_S   mean=-0.1922  std=9.3040
  PnL_VP  mean=0.0033  std=0.4872
  C_T     mean=3.3017  std=4.7962
  dVP     mean abs=0.007365
  dS      mean abs=0.724401
[ 1/15] N=5000 clean s0         Mean Error=3.7227  (32s)


N=5000 S s0 train:   0%|          | 0/160 [00:00<?, ?it/s]  loss_c=0.0000  loss_a=0.0000  p0=3.7246

...

  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [GRAD] last clean epoch grad norm = nan  p0_clean = 2.7396
  [SANITY] p0_clean = 2.7396
  [SANITY] Train Mean Error = 3.5554  (should be < C_T mean 1.76 if learning)
  [SANITY] Train ES_0.5     = 5.3087
  h[:,:,0] mean=0.3767 std=0.3349  (stock holdings)
  h[:,:,1] mean=5.5160 std=5.0576  (varswap holdings)
  PnL_S   mean=0.0063  std=4.7689
  PnL_VP  mean=-0.1120  std=1.4752
  C_T     mean=3.3017  std=4.7962
  dVP     mean abs=0.007365
  dS     

N=100000 S s0 train:   0%|          | 0/160 [00:00<?, ?it/s]  loss_c=0.0000  loss_a=0.0000  p0=3.7246

  [GRAD] last clean epoch grad norm = 7.688574  p0_clean = 3.0610
  [GRAD] last clean epoch grad norm = 7.632532  p0_clean = 3.0611
  [GRAD] last clean epoch grad norm = 7.688215  p0_clean = 3.0613
  [GRAD] last clean epoch grad norm = 7.770779  p0_clean = 3.0614
  [GRAD] last clean epoch grad norm = 7.688075  p0_clean = 3.0616
  [GRAD] last clean epoch grad norm = 7.623046  p0_clean = 3.0617
  [GRAD] last clean epoch grad norm = 7.753989  p0_clean = 3.0619
  [GRAD] last clean epoch grad norm = 7.617189  p0_clean = 3.0621
  [GRAD] last clean epoch grad norm = 7.740680  p0_clean = 3.0622
  [GRAD] last clean epoch grad norm = 7.734541  p0_clean = 3.0623
  [LOSS] clean mean=3.5746  att mean=3.8313  diff=0.2567
  [LOSS] clean mean=3.6175  att mean=3.6195  diff=0.0020
  [LOSS] clean mean=3.6009  att mean=3.6027  diff=0.0019
  [LOSS] clean mean=3.6167  att mean=3.6183  diff=0.0017
  [LOSS] clean mean=3.5474  att mean=3.5483  diff=0.0009
  [LOSS] clean mean=3.6007  att mean=3.6007  diff=0.000

N=100000 SV s0 train:   0%|          | 0/160 [00:00<?, ?it/s]  loss_c=0.0000  loss_a=0.0000  p0=3.7246

  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0067
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0069
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0071
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0073
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0074
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0076
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0078
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0079
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0081
  [GRAD] last clean epoch grad norm = nan  p0_clean = 3.0083
  [LOSS] clean mean=3.5879  att mean=3.5879  diff=0.0000
  [LOSS] clean mean=3.5761  att mean=3.5761  diff=0.0000
  [LOSS] clean mean=3.5326  att mean=3.5326  diff=0.0000
  [LOSS] clean mean=3.5911  att mean=3.5911  diff=0.0000
  [LOSS] clean mean=3.5547  att mean=3.5547  diff=0.0000
  [LOSS] clean mean=3.5721  att mean=3.5721  diff=0.0000
  [LOSS] clean mean=3.6048  att mean=3.6048  dif

In [ ]:
# ── OOD Evaluation ────────────────────────────────────────────────────────────
np.random.seed(0)

ood_sim     = OutOfSampleHestonSimulator(base_params, variation=0.1)
results_ood = {}

total_ood = sum(min(M_TRAIN // N, N_SEEDS) for N in N_SIZES) * len(METHODS)
print(f"OOD evaluation: {total_ood} networks x {N_OOD_CFGS} configs each\n")
t_ood = time.perf_counter()

for N in N_SIZES:
    n_runs = min(M_TRAIN // N, N_SEEDS)
    for method in METHODS:
        for s in range(n_runs):
            if (N, method, s) not in stored_nets:
                continue
            net = stored_nets[(N, method, s)]
            ood_es_list = []
            for cfg in range(N_OOD_CFGS):
                S_o, V_o, VP_o, _ = ood_sim.simulate(M=M_OOD)
                VP_o = VP_o * VP_SCALE   # match the scaling used during training
                err = compute_hedging_errors(net, S_o, V_o, VP_o)
                ood_es_list.append(empirical_ES(err, alpha=ALPHA_CVAR))
            results_ood[(N, method, s)] = float(np.mean(ood_es_list))

        vals = [f"{results_ood.get((N, method, s), float('nan')):.4f}"
                for s in range(n_runs)]
        print(f"  N={N:6d}  {method:6s}  OOD ES = {' | '.join(vals)}")

print(f"\nOOD done in {(time.perf_counter()-t_ood)/60:.1f} min")

In [ ]:
# ── Plot Figure 1 — OOS and OOD Performance ───────────────────────────────────
from pathlib import Path
from IPython.display import display

COLORS = {'clean': 'steelblue', 'S': 'darkorange', 'SV': 'seagreen'}
LABELS = {'clean': 'Clean', 'S': 'S-Attack', 'SV': 'SV-Attack'}


def _plot_panel(ax, results_dict: dict, title: str) -> None:
    """Draw one panel: mean line + min-max shaded band across seeds."""
    for method in METHODS:
        means, mins_, maxs_ = [], [], []
        for N in N_SIZES:
            n_runs = min(M_TRAIN // N, N_SEEDS)
            vals   = [results_dict[(N, method, s)]
                      for s in range(n_runs) if (N, method, s) in results_dict]
            if vals:
                means.append(float(np.mean(vals)))
                mins_.append(float(np.min(vals)))
                maxs_.append(float(np.max(vals)))
            else:
                means.append(float('nan'))
                mins_.append(float('nan'))
                maxs_.append(float('nan'))

        ax.plot(N_SIZES, means, color=COLORS[method], label=LABELS[method],
                linewidth=2.0, marker='o', markersize=4)
        ax.fill_between(N_SIZES, mins_, maxs_, alpha=0.20, color=COLORS[method])

    ax.set_xscale('log')
    ax.set_xlabel('Training Samples (N)', fontsize=12)
    ax.set_ylabel(r'Hedging Loss (ES$_{0.95}$)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
_plot_panel(axes[0], results_oos, '(a) Out-of-sample performance')
_plot_panel(axes[1], results_ood, '(b) Out-of-distribution performance')

fig.suptitle(
    'Comparative Hedging Performance under Heston Model\n'
    'Shaded regions: min-max range across seeds / partitions',
    fontsize=12, y=1.03,
)
plt.tight_layout()
fig.savefig(Path('..') / 'results' / 'heston_figure1_adversarial.png',
            dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)

In [ ]:
# ── Summary Table ──────────────────────────────────────────────────────────────
N_small = N_SIZES[0]
N_large = N_SIZES[-1]

def _get_mean(results_dict, N, method):
    n_runs = min(M_TRAIN // N, N_SEEDS)
    vals = [results_dict.get((N, method, s), float('nan')) for s in range(n_runs)]
    return float(np.nanmean(vals))

print(f"\n{'':24s}  {'N='+str(N_small):>10s}  {'N='+str(N_large):>10s}")
print('-' * 48)
for label, results in [('OOS', results_oos), ('OOD', results_ood)]:
    for method in METHODS:
        v_small = _get_mean(results, N_small, method)
        v_large = _get_mean(results, N_large, method)
        print(f"  {label} {LABELS[method]:20s}  {v_small:10.4f}  {v_large:10.4f}")
    print()

oos_clean  = _get_mean(results_oos, N_small, 'clean')
oos_sv_atk = _get_mean(results_oos, N_small, 'SV')
if oos_clean > 0 and not np.isnan(oos_sv_atk):
    pct = 100 * (oos_clean - oos_sv_atk) / oos_clean
    print(f"SV-Attack vs Clean at N={N_small}: {pct:.1f}% lower OOS loss"
          f"  ({oos_sv_atk:.3f} vs {oos_clean:.3f})")
    print('(Paper reports ~54% lower at N=5,000 for the full experiment)')


                              N=5000    N=100000
------------------------------------------------
  OOS Clean                     3.6161      2.7729
  OOS S-Attack                  3.5000      2.1730
  OOS SV-Attack                 3.5658      2.1788

  OOD Clean                     3.0637      2.6962
  OOD S-Attack                  2.9894      2.8576
  OOD SV-Attack                 3.0363      2.8476

SV-Attack vs Clean at N=5000: 1.4% lower OOS loss  (3.566 vs 3.616)
(Paper reports ~54% lower at N=5,000 for the full experiment)
